# 04 — Feature-based model: add techniques one at a time (Store Sales)

**Goal of this notebook:** start from the Stage 2 deterministic model (trend + Fourier) and add feature groups **one at a time** — holidays, calendar, promotions & oil, then lags — re-scoring on the same 16-day holdout after each so the iteration log shows *which* technique earns the margin (FR-010, SC-005).

**The model.** `LinearFeatureModel` (in `src/models.py`) generalizes the Stage 2 per-series `LinearRegression` to accept a prebuilt feature matrix. It keeps the same ragged-history defenses (active-history fit, annual-Fourier drop for short series) and adds a lag warm-up NaN-drop. Granularity stays **per-series** — the global-vs-per-group decision is Stage 4 (T033).

**Leak safety.** Every feature is built **once over the full history** and split by date. That is safe because the holdout span (16 days) equals `base_lag` (16): the farthest holdout day's `lag_16` reaches back to the last training day, never into the holdout. We assert this each stage with `assert_no_leak(..., base_lag=HORIZON_DAYS)` — the guard T029a exists for.

Run it top-to-bottom from a fresh kernel.

## 0. Setup

Put the repo root on the import path so `src/` is importable from `notebooks/`. All loading, feature-building, the model, scoring, and logging live in `src/` — the notebook only composes and narrates.

In [1]:
import sys
from pathlib import Path

here = Path.cwd()
repo_root = here if (here / "src").exists() else here.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd

import src.data as data
import src.features as features
import src.models as models
import src.validation as validation

print("repo root:", repo_root)

repo root: D:\AIO\M01\Conquer\TimeSeries


## 1. Load and reindex (gap-free)

Load `train.csv` and reindex every `(store_nbr, family)` series onto a complete daily calendar (closed days -> `sales = 0`). Gap-free is essential here: lag features count *rows* as days, and the Fourier terms read each row's calendar position — a missing day would corrupt both.

In [2]:
df = data.reindex_series_gapfree(data.load_train())
validation.assert_gapfree(df)

KEY = ["store_nbr", "family"]
dates = pd.DatetimeIndex(np.sort(df["date"].unique()))
print(f"gap-free rows: {len(df):,}")
print(f"series:        {df[KEY].drop_duplicates().shape[0]:,}")
print(f"days:          {len(dates)}  ({dates.min().date()} -> {dates.max().date()})")

gap-free rows: 3,008,016
series:        1,782
days:          1688  (2013-01-01 -> 2017-08-15)


## 2. Build each feature group

Each builder from `src/features.py` produces one group, all **knowable in advance** (date-only, forward-filled, or lagged by >= 16 days):

- **deterministic** (T023) — const, trend, weekly + annual Fourier (by date)
- **calendar** (T027) — day-of-week, month, payday window, earthquake flag (by date)
- **oil** (T028) — forward-filled WTI price (by date)
- **holidays** (T026) — effective, locale-scoped holiday flags (by store x date)
- **lags** (T029) — sales lag/rolling + lagged transactions, all anchored at `shift(16)` (by series x date)

`onpromotion` (T028) is already in `df` and used **contemporaneously** (it ships in `test.csv` for the horizon) — no lag.

In [3]:
det  = features.make_deterministic_features(dates).rename_axis("date").reset_index()
cal  = features.make_calendar_features(dates).rename_axis("date").reset_index()
oil  = features.make_oil_features(data.load_oil(), dates).rename_axis("date").reset_index()
hol  = features.make_holiday_features(data.load_holidays(), data.load_stores())
lags = features.make_lag_features(df, data.load_transactions())

DET_COLS  = list(det.columns[1:])
CAL_COLS  = list(cal.columns[1:])
HOL_COLS  = ["is_holiday", "is_work_day", "is_national_holiday"]
PROMO_OIL = ["onpromotion", "oil"]
LAG_COLS  = [c for c in lags.columns if c not in KEY + ["date"]]

print("deterministic:", DET_COLS)
print("calendar     :", CAL_COLS)
print("holiday      :", HOL_COLS)
print("promo & oil  :", PROMO_OIL)
print("lags         :", LAG_COLS)

deterministic: ['const', 'trend', 'sin(1,freq=W-SUN)', 'cos(1,freq=W-SUN)', 'sin(2,freq=W-SUN)', 'cos(2,freq=W-SUN)', 'sin(3,freq=W-SUN)', 'cos(3,freq=W-SUN)', 'sin(1,freq=YE-DEC)', 'cos(1,freq=YE-DEC)', 'sin(2,freq=YE-DEC)', 'cos(2,freq=YE-DEC)', 'sin(3,freq=YE-DEC)', 'cos(3,freq=YE-DEC)', 'sin(4,freq=YE-DEC)', 'cos(4,freq=YE-DEC)']
calendar     : ['dayofweek', 'month', 'is_payday', 'is_payday_window', 'is_earthquake']
holiday      : ['is_holiday', 'is_work_day', 'is_national_holiday']
promo & oil  : ['onpromotion', 'oil']
lags         : ['sales_lag_16', 'sales_lag_17', 'sales_lag_18', 'sales_roll_7_lag_16_mean', 'sales_roll_14_lag_16_mean', 'sales_roll_28_lag_16_mean', 'transactions_lag_16']


## 3. Compose one tidy feature matrix

Merge every group onto the gap-free grid by its natural key: date-only groups by `date`, holidays by `(store_nbr, date)` (absent store-days -> 0, an ordinary day), lags by `(store_nbr, family, date)`. Building once and splitting later keeps the trend counter and lag history continuous.

In [4]:
matrix = (
    df[KEY + ["date", "sales", "onpromotion"]]
    .merge(det, on="date")
    .merge(cal, on="date")
    .merge(oil, on="date")
    .merge(hol, on=["store_nbr", "date"], how="left")
)
matrix[HOL_COLS] = matrix[HOL_COLS].fillna(0).astype("int8")
matrix = matrix.merge(lags, on=KEY + ["date"], how="left")

print("composed matrix:", matrix.shape)
matrix.head(3)

composed matrix: (3008016, 37)


,store_nbr,family,date,sales,onpromotion,const,trend,"sin(1,freq=W-SUN)","cos(1,freq=W-SUN)","sin(2,freq=W-SUN)",...,is_holiday,is_work_day,is_national_holiday,sales_lag_16,sales_lag_17,sales_lag_18,sales_roll_7_lag_16_mean,sales_roll_14_lag_16_mean,sales_roll_28_lag_16_mean,transactions_lag_16
0,1,AUTOMOTIVE,2013-01-01,0.0,0,1.0,1.0,0.781831,0.623490,0.974928,...,1,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,AUTOMOTIVE,2013-01-02,2.0,0,1.0,2.0,0.974928,-0.222521,-0.433884,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,AUTOMOTIVE,2013-01-03,3.0,0,1.0,3.0,0.433884,-0.900969,-0.781831,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Holdout split

Carve off the last 16 days (2017-07-31 -> 2017-08-15) — the same window as every prior stage. `train_m` is everything strictly before; we fit on it and score on the held-out actuals.

In [5]:
holdout_start = df["date"].max() - pd.Timedelta(days=validation.HORIZON_DAYS - 1)
train_m = matrix[matrix["date"] < holdout_start]
hold_m = matrix[matrix["date"] >= holdout_start]
truth = hold_m[KEY + ["date", "sales"]]

print(f"train:   {train_m['date'].min().date()} -> {train_m['date'].max().date()}  ({len(train_m):,} rows)")
print(f"holdout: {hold_m['date'].min().date()} -> {hold_m['date'].max().date()}  ({len(hold_m):,} rows)")

train:   2013-01-01 -> 2017-07-30  (2,979,504 rows)
holdout: 2017-07-31 -> 2017-08-15  (28,512 rows)


## 5. Add feature groups one at a time

`run_stage` is pure orchestration: it (1) runs the leak guard on the *fitted* rows — they must be strictly before the holdout, and every sales/transactions lag must reach back >= 16 days; (2) fits `LinearFeatureModel` on the chosen columns; (3) scores RMSLE on the holdout; (4) logs the honest number via `log_iteration`. The deterministic row is shown for reference (already logged in Stage 2); the four additions are logged here.

In [6]:
def run_stage(name, cols, *, log):
    validation.assert_no_leak(
        train_m[KEY + ["date"] + cols], holdout_start,
        strict=True, base_lag=validation.HORIZON_DAYS,
    )
    model = models.LinearFeatureModel(cols).fit(train_m)
    preds = model.predict(hold_m)
    scored = truth.merge(preds, on=KEY + ["date"])
    score = validation.rmsle(scored["sales"], scored["sales_pred"])
    if log:
        delta = validation.log_iteration(
            f"feature model: {name}", score,
            notes=f"{len(cols)} features; per-series LinearRegression, log1p; no-leak base_lag=16",
        )
        print(f"{name:22s} RMSLE={score:.5f}  ->  {delta}")
    else:
        print(f"{name:22s} RMSLE={score:.5f}  (reference; logged in Stage 2)")
    return score

STAGES = [
    ("deterministic only",  DET_COLS,                                             False),
    ("+ holidays",          DET_COLS + HOL_COLS,                                  True),
    ("+ calendar",          DET_COLS + HOL_COLS + CAL_COLS,                       True),
    ("+ promotions & oil",  DET_COLS + HOL_COLS + CAL_COLS + PROMO_OIL,           True),
    ("+ lags",              DET_COLS + HOL_COLS + CAL_COLS + PROMO_OIL + LAG_COLS, True),
]
scores = {name: run_stage(name, cols, log=log) for name, cols, log in STAGES}

deterministic only     RMSLE=0.62188  (reference; logged in Stage 2)


+ holidays             RMSLE=0.62305  ->  +0.00601 (worse)


+ calendar             RMSLE=0.60717  ->  -0.00987 (better)


+ promotions & oil     RMSLE=0.58226  ->  -0.02491 (better)


+ lags                 RMSLE=0.50991  ->  -0.07235 (better)


## 6. Compare the best feature model to the baseline

Recompute the seasonal-naive baseline on the same split and measure the relative RMSLE reduction of the full feature model — the SC-003 yardstick (target >= 10%).

In [7]:
base_preds = models.seasonal_naive_predict(train_m, validation.HORIZON_DAYS)
base_scored = truth.merge(base_preds, on=KEY + ["date"])
baseline = validation.rmsle(base_scored["sales"], base_scored["sales_pred"])

best = scores["+ lags"]
rel = (baseline - best) / baseline * 100
print(f"baseline seasonal-naive : {baseline:.5f}")
print(f"best feature model      : {best:.5f}")
print(f"relative improvement    : {rel:+.1f}%   (SC-003 target: >= 10%)")

baseline seasonal-naive : 0.61704
best feature model      : 0.50991
relative improvement    : +17.4%   (SC-003 target: >= 10%)


## 7. Ablation — does every feature group earn its place?

Section 5 *added* groups cumulatively; here we *remove* one group at a time from the full set to isolate each group's marginal contribution. Because `LinearFeatureModel` takes any column list, **feature selection is just choosing `cols`** — nothing else changes (the leak guard, active-history fit, and annual-Fourier drop still apply to whatever columns are present). So you can freely add or drop any group.

Read the **contribution** = `RMSLE_without − RMSLE_full`: **positive** means the group *helps* (removing it makes the score worse); **negative** means the group actually *hurts* and is better dropped. We split `onpromotion` and `oil` out of the combined "promo & oil" stage so each is judged on its own.

In [8]:
def score_cols(cols):
    """Fit LinearFeatureModel on any column list and return its holdout RMSLE (leak-guarded)."""
    validation.assert_no_leak(
        train_m[KEY + ["date"] + cols], holdout_start,
        strict=True, base_lag=validation.HORIZON_DAYS,
    )
    preds = models.LinearFeatureModel(cols).fit(train_m).predict(hold_m)
    scored = truth.merge(preds, on=KEY + ["date"])
    return validation.rmsle(scored["sales"], scored["sales_pred"])

FULL = DET_COLS + HOL_COLS + CAL_COLS + PROMO_OIL + LAG_COLS
GROUPS = {
    "holidays":   HOL_COLS,
    "calendar":   CAL_COLS,
    "promotions": ["onpromotion"],
    "oil":        ["oil"],
    "lags":       LAG_COLS,
}

full_score = scores["+ lags"]  # the full set, already fitted in Section 5
print(f"full feature model: {full_score:.5f}\n")
print("leave-one-group-out  (contribution = RMSLE_without - RMSLE_full):")
print("  positive => group HELPS (removing it hurts) | negative => group HURTS (drop it)\n")
for name, group in GROUPS.items():
    without = score_cols([c for c in FULL if c not in group])
    contrib = without - full_score
    tag = "helps" if contrib > 1e-4 else ("HURTS -> drop" if contrib < -1e-4 else "neutral")
    print(f"  - {name:11s} without it: {without:.5f}   contribution {contrib:+.5f}  ({tag})")

best = [c for c in FULL if c != "oil"]
print(f"\nbest linear set (full minus oil): {score_cols(best):.5f}  vs full {full_score:.5f}")

full feature model: 0.50991

leave-one-group-out  (contribution = RMSLE_without - RMSLE_full):
  positive => group HELPS (removing it hurts) | negative => group HURTS (drop it)



  - holidays    without it: 0.50907   contribution -0.00085  (HURTS -> drop)


  - calendar    without it: 0.51357   contribution +0.00366  (helps)


  - promotions  without it: 0.54676   contribution +0.03685  (helps)


  - oil         without it: 0.49365   contribution -0.01626  (HURTS -> drop)


  - lags        without it: 0.58226   contribution +0.07235  (helps)



best linear set (full minus oil): 0.49365  vs full 0.50991


## Takeaway

Adding features one at a time, on the same 16-day holdout, tells an honest story:

| Added | Holdout RMSLE | vs baseline 0.617 |
|---|---|---|
| deterministic only | ~0.622 | parity |
| + holidays | ~0.623 | slightly **worse** |
| + calendar | ~0.607 | better |
| + promotions & oil | ~0.582 | better |
| **+ lags** | **~0.510** | **~17% better** |

- **Holidays alone slightly hurt.** A single `is_holiday` flag conflates demand-boosting holidays with closure holidays (many national holidays *close* stores), so a plain linear term struggles — an honest non-improvement, kept in the log (FR-010). It may still help once it interacts with other terms in the Stage-4 tree.
- **Calendar and promotions help steadily** — the payday window / earthquake flag and especially the promotion signal add real, separable lift.
- **Lags deliver the big jump.** They recover the *recent level* that pure trend + seasonality smooths away — exactly the edge the seasonal-naive baseline had. This is what pushes the model past the **SC-003 >= 10%** bar.
- **The ablation (Section 7) shows not every feature earns its place.** Holidays are ~neutral, but **oil actually *hurts*** the linear model — consistent with the EDA finding that its sales correlation is a spurious trend artifact. The best linear set is roughly **full minus oil** (~0.494). We keep oil out here and let the Stage-4 tree decide whether it can extract any non-linear signal.
- **No leakage at any stage:** the `base_lag = 16` guard (T029a) passed for every feature set, including every ablation fit.

**Next (Stage 4 — `05_hybrid.ipynb`):** fit XGBoost on this linear model's residuals (the hybrid), decide global-vs-per-group empirically (T033), then produce the valid submission and target leaderboard RMSLE <= 0.45 (SC-004).